# Notebook 02 — Shared Text Preprocessing
**Week 2**: Clean discharge-note text, build TF-IDF features (for Model A) and prepare tokenizer inputs (for Model B). Outputs re-usable label binarizer and feature matrices saved locally.

In [1]:
%pip install pandas pyarrow scikit-learn transformers 

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typer-0.24.2-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annot

## 1. Config

In [2]:
import pandas as pd
import numpy as np
import re
import os
import pickle

DATA_DIR = '../datasets/processed'

# Label filtering — Top-K most frequent codes
TOP_K_LABELS = 50            # Set to 50, 500, or None for full label set

# TF-IDF config
TFIDF_MAX_FEATURES = 50_000  # reduced from 150k for faster training
TFIDF_NGRAM_RANGE  = (1, 2)
TFIDF_SUBLINEAR_TF = True

# Transformer tokenizer (for Model B reference)
TRANSFORMER_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_SEQ_LEN       = 512

## 2. Load cohort splits

In [3]:
train_df = pd.read_parquet(f'{DATA_DIR}/cohort_train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/cohort_val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/cohort_test.parquet')
vocab    = pd.read_csv(f'{DATA_DIR}/label_vocab.csv')['icd_code'].tolist()

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print(f'Label vocab size: {len(vocab)}')

# Restore list from pipe-separated string
for df_ in [train_df, val_df, test_df]:
    df_['icd_codes'] = df_['icd_codes_str'].str.split('|')

Train: 85437  Val: 18195  Test: 18672
Label vocab size: 7940


## 3. Text cleaning

In [4]:
# De-identification placeholder patterns common in MIMIC
_DEID_RE = re.compile(
    r'\[\*\*[^\]]*\*\*\]'   # [** ... **]  placeholders
)
_WHITESPACE_RE = re.compile(r'[\s\n\r\t]+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = _DEID_RE.sub(' ', text)          # remove de-id tokens
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s.,;:\-/]', ' ', text)  # keep alphanumeric + basic punctuation
    text = _WHITESPACE_RE.sub(' ', text).strip()
    return text

for df_ in [train_df, val_df, test_df]:
    df_['clean_text'] = df_['text'].apply(clean_text)

print('Sample cleaned note (first 300 chars):')
print(train_df['clean_text'].iloc[0][:300])

Sample cleaned note (first 300 chars):
name: . unit no: admission date: discharge date: date of birth: sex: f service: medicine allergies: demerol / latex attending: . major surgical or invasive procedure: : push enteroscopy and colonoscopy attach pertinent results: admission labs -------------------- 12:16am blood wbc-4.8 rbc-2.04 hgb-5


## 4. Label binarization (MultiLabelBinarizer)

In [5]:
from sklearn.preprocessing import MultiLabelBinarizer

# First pass: compute training frequency for all labels
mlb_full = MultiLabelBinarizer(classes=vocab)
mlb_full.fit([vocab])
Y_train_full = mlb_full.transform(train_df['icd_codes']).astype(np.float32)
freq = Y_train_full.sum(axis=0)

# Select top-K labels by training frequency
if TOP_K_LABELS is not None:
    top_idx = np.argsort(freq)[::-1][:TOP_K_LABELS]
    vocab_selected = sorted([vocab[i] for i in top_idx])
    coverage = freq[top_idx].sum() / freq.sum()
    print(f'Selected top-{TOP_K_LABELS} labels (coverage: {coverage:.1%} of all assignments)')
else:
    vocab_selected = vocab

mlb = MultiLabelBinarizer(classes=vocab_selected)
mlb.fit([vocab_selected])
Y_train = mlb.transform(train_df['icd_codes']).astype(np.float32)
Y_val   = mlb.transform(val_df['icd_codes']).astype(np.float32)
Y_test  = mlb.transform(test_df['icd_codes']).astype(np.float32)

print(f'Y_train shape: {Y_train.shape}   (notes × labels)')
print(f'Label density (train): {Y_train.mean():.4f}')

# Save binarizer
with open(f'{DATA_DIR}/mlb.pkl', 'wb') as f:
    pickle.dump(mlb, f)

# Save full vocab + frequencies for later analysis
label_freq_df = pd.DataFrame({'icd_code': vocab, 'train_freq': freq})
label_freq_df.to_csv(f'{DATA_DIR}/label_freq.csv', index=False)
print('Saved mlb.pkl and label_freq.csv')

/opt/homebrew/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['00160J2', '00163J4', '00500ZZ', '00560ZZ', '005C0ZZ', '005K0ZZ', '005T0ZZ', '008Q0ZZ', '008X0ZZ', '009000Z', '00900ZZ', '009030Z', '00903ZZ', '00904ZZ', '00920ZZ', '009300Z', '00930ZX', '00930ZZ', '009330Z', '00933ZZ', '00950ZZ', '00953ZX', '00960ZZ', '00963ZX', '00963ZZ', '009640Z', '009700Z', '009730Z', '00973ZZ', '009C0ZZ', '009T00Z', '009T0ZZ', '009T30Z', '009T3ZX', '009U0ZX', '009W00Z', '009W0ZZ', '009X3ZX', '009X3ZZ', '009Y30Z', '009Y3ZX', '009Y3ZZ', '00B03ZZ', '00B04ZZ', '00B13ZX', '00B14ZZ', '00B23ZX', '00B24ZZ', '00B60ZX', '00B74ZX', '00B80ZX', '00B83ZX', '00B90ZX', '00B90ZZ', '00B93ZX', '00BB0ZX', '00BB0ZZ', '00BB3ZX', '00BG0ZX', '00BK0ZZ', '00BN0ZZ', '00BN4ZX', '00BN4ZZ', '00BQ0ZZ', '00BQ4ZZ', '00BR0ZZ', '00BW0ZX', '00BW3ZX', '00BX3ZX', '00C03ZZ', '00C10ZZ', '00C20ZZ', '00C50ZZ', '00C54ZZ', '00C80ZZ', '00CT0ZZ', '00CW0ZZ', '00CX0ZZ', '00CY0ZZ', '00D20ZZ', '00H002

Selected top-50 labels (coverage: 32.5% of all assignments)
Y_train shape: (85437, 50)   (notes × labels)
Label density (train): 0.1017
Saved mlb.pkl and label_freq.csv


/opt/homebrew/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['00160J2', '00160J6', '00163J4', '00163J6', '00164J6', '00500ZZ', '00560ZZ', '005C0ZZ', '005K0ZZ', '005T0ZZ', '008Q0ZZ', '008Q4ZZ', '008X0ZZ', '009000Z', '00900ZZ', '009030Z', '00903ZZ', '00904ZZ', '00920ZZ', '009300Z', '00930ZX', '00930ZZ', '009330Z', '00933ZZ', '009400Z', '00940ZZ', '009430Z', '00943ZZ', '00950ZZ', '00953ZX', '009600Z', '00960ZZ', '009630Z', '00963ZX', '00963ZZ', '009640Z', '009700Z', '00970ZZ', '009730Z', '00973ZZ', '009C0ZZ', '009T00Z', '009T0ZZ', '009T30Z', '009T3ZX', '009U00Z', '009U0ZX', '009U0ZZ', '009U30Z', '009U3ZX', '009U3ZZ', '009W00Z', '009W0ZZ', '009X3ZX', '009X3ZZ', '009Y30Z', '009Y3ZX', '009Y3ZZ', '00B00ZX', '00B00ZZ', '00B03ZX', '00B03ZZ', '00B04ZZ', '00B10ZX', '00B10ZZ', '00B13ZX', '00B14ZZ', '00B20ZX', '00B20ZZ', '00B23ZX', '00B24ZZ', '00B60ZX', '00B60ZZ', '00B70ZX', '00B70ZZ', '00B73ZX', '00B74ZX', '00B80ZX', '00B83ZX', '00B90ZX', '00B90Z

## 5. TF-IDF vectorisation (Model A features)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    sublinear_tf=TFIDF_SUBLINEAR_TF,
    min_df=3,
    strip_accents='unicode',
    analyzer='word',
)

print('Fitting TF-IDF on train set...')
X_train_tfidf = vectorizer.fit_transform(train_df['clean_text'])
X_val_tfidf   = vectorizer.transform(val_df['clean_text'])
X_test_tfidf  = vectorizer.transform(test_df['clean_text'])

print(f'X_train_tfidf shape: {X_train_tfidf.shape}')

# Save
sp.save_npz(f'{DATA_DIR}/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz(f'{DATA_DIR}/X_val_tfidf.npz',   X_val_tfidf)
sp.save_npz(f'{DATA_DIR}/X_test_tfidf.npz',  X_test_tfidf)
np.save(f'{DATA_DIR}/Y_train.npy', Y_train)
np.save(f'{DATA_DIR}/Y_val.npy',   Y_val)
np.save(f'{DATA_DIR}/Y_test.npy',  Y_test)

with open(f'{DATA_DIR}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('All features and labels saved.')

Fitting TF-IDF on train set...
X_train_tfidf shape: (85437, 50000)
All features and labels saved.


## 6. Save cleaned text splits (for transformer notebook)

In [7]:
for name, df_ in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df_[['subject_id', 'hadm_id', 'clean_text', 'icd_codes_str', 'split']].to_parquet(
        f'{DATA_DIR}/cohort_{name}_clean.parquet', index=False)
print('Clean text parquets saved.')

Clean text parquets saved.
